# 03 — Gene Regulatory Network Construction

Builds cell-type-specific gene regulatory networks with CellOracle, using the
mouse scATAC-seq atlas as the base network. The Yamanaka factors are retained in
the feature set irrespective of variance ranking, as they are required as
perturbation targets in later notebooks; standard variance-based selection
excludes Myc and Oct4 (Pou5f1).

In [ ]:
import sys
!{sys.executable} -m pip install --no-deps git+https://github.com/morris-lab/CellOracle.git
!{sys.executable} -m pip install "numpy==1.26.4" anndata scanpy genomepy gimmemotifs goatools igraph jupyter louvain pybedtools velocyto
!{sys.executable} -m pip install fa2-modified

from google.colab import drive
drive.mount('/content/drive')

  Cloning https://github.com/morris-lab/CellOracle.git to /tmp/pip-req-build-xpklkopw
  Running command git clone --filter=blob:none --quiet https://github.com/morris-lab/CellOracle.git /tmp/pip-req-build-xpklkopw
  Resolved https://github.com/morris-lab/CellOracle.git to commit 7948870a3b70f7d228e54734e1fa9ed3291fc23b
  Preparing metadata (setup.py) ... done
  Created wheel for celloracle: filename=celloracle-0.22.0-py3-none-any.whl size=12372847 sha256=919311c948a8483855d95d49b2b6d87ae4ab5c04f6cdf29f2987a10be795af96
  Stored in directory: /tmp/pip-ephem-wheel-cache-jz1ygeia/wheels/cc/aa/03/95b3761e1aa553e81e2506c7f64cf5878e46cad7f5eeffd09d
Successfully built celloracle
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 55.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 46

In [ ]:
import celloracle as co
print(co.__version__)
import velocyto
print("velocyto ok")

21:31:10 - INFO - Creating new config.
21:31:10 - INFO - Using included version of AMD.
21:31:10 - INFO - Using included version of BioProspector.
21:31:10 - INFO - Using included version of ChIPMunk.
21:31:10 - WARNING - DiNAMO not found. To include it you will have to install it.
21:31:10 - WARNING - DREME not found. To include it you will have to install it.
21:31:10 - WARNING - GADEM not found. To include it you will have to install it.
21:31:10 - INFO - Using included version of HMS.
21:31:10 - WARNING - Homer not found. To include it you will have to install it.
21:31:10 - INFO - Using included version of Improbizer.
21:31:10 - INFO - Using included version of MDmodule.
21:31:10 - WARNING - MEME not found. To include it you will have to install it.
21:31:10 - WARNING - MEMEW not found. To include it you will have to install it.
21:31:10 - INFO - Using included version of MotifSampler.
21:31:10 - INFO - Using included version of Posmo.
21:31:10 - WARNING - ProSampler not found. To

0.22.0
velocyto ok


In [ ]:
import scanpy as sc
import anndata as ad

# the full-gene checkpoint retains Myc and Pou5f1; the annotated one has cell_type
adata_full = ad.read_h5ad("/content/drive/MyDrive/roux_project/msc_clean_raw.h5ad")
adata_annot = ad.read_h5ad("/content/drive/MyDrive/roux_project/msc_annotated.h5ad")

print("full genes:", adata_full.shape, "| Myc present?", 'Myc' in adata_full.var_names)
print("annotated:", adata_annot.shape)
print("annotated cells subset of full?", adata_annot.obs_names.isin(adata_full.obs_names).all())

# subset the full-gene data to the annotated cells and transfer the labels
adata = adata_full[adata_annot.obs_names].copy()
adata.obs['cell_type'] = adata_annot.obs['cell_type'].values

print("merged:", adata.shape)
print("all four factors present?", all(g in adata.var_names for g in ['Klf4','Myc','Sox2','Pou5f1']))
print("raw_count layer?", 'raw_count' in adata.layers)

full genes: (10018, 19321) | Myc present? True
annotated: (9977, 2000)
annotated cells subset of full? True
merged: (9977, 19321)
all four factors present? True
raw_count layer? True


In [ ]:
adata.X = adata.layers['raw_count'].copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=2000)

# retain the Yamanaka factors regardless of variance ranking
yamanaka = ['Klf4', 'Myc', 'Sox2', 'Pou5f1']
for g in yamanaka:
    adata.var.loc[g, 'highly_variable'] = True

print("HVG count:", adata.var['highly_variable'].sum())
print("all four flagged?", all(adata.var.loc[g, 'highly_variable'] for g in yamanaka))

# stash the full normalised matrix, subset to HVG, scale, restore raw counts for CellOracle
adata.raw = adata
adata = adata[:, adata.var['highly_variable']].copy()
sc.pp.scale(adata, max_value=10)
adata.X = adata.layers['raw_count'].copy()

print("subset to HVG:", adata.shape, "| all 4 factors in?", all(g in adata.var_names for g in yamanaka))
print("X_umap present?", 'X_umap' in adata.obsm)

HVG count: 2002
all four flagged? True


/usr/lib/python3.13/functools.py:934: UserWarning: zero-centering a sparse array/matrix densifies it.
  return dispatch(args[0].__class__)(*args, **kw)


subset to HVG: (9977, 2002) | all 4 factors in? True
X_umap present? True


In [ ]:
import celloracle as co

oracle = co.Oracle()
oracle.import_anndata_as_raw_count(
    adata=adata,
    cluster_column_name="cell_type",
    embedding_name="X_umap"
)

# the dataset provides no chromatin accessibility, so the built-in mouse
# scATAC-seq atlas is used as the base regulatory network
base_GRN = co.data.load_mouse_scATAC_atlas_base_GRN()
oracle.import_TF_data(TF_info_matrix=base_GRN)
print("base GRN imported")

/usr/local/lib/python3.13/dist-packages/celloracle/trajectory/oracle_utility.py:28: UserWarning: Color information for the cell_type is not found in the anndata. CellOracle is plotting the clustering data, cell_type, to create color data.
  warnings.warn(message, UserWarning)


Data not found in the local folder. Loading data from github. Data will be saved at /root/celloracle_data/TFinfo_data


  0%|          | 0.00/9.01M [00:00<?, ?B/s]

base GRN imported


In [ ]:
oracle.perform_PCA()
oracle.knn_imputation(
    n_pca_dims=50, k=100, balanced=True, b_sight=800, b_maxl=400, n_jobs=4
)

links = oracle.get_links(
    cluster_name_for_GRN_unit="cell_type",
    alpha=10,
    verbose_level=1
)
print("networks inferred")

  0%|          | 0/9 [00:00<?, ?it/s]

networks inferred


In [ ]:
links.filter_links(p=0.001, weight="coef_abs", threshold_number=2000)
print(links.links_dict.keys())
links.filtered_links["Fibroblast/stromal"].head(10)

dict_keys(['Adipogenic', 'Fibroblast/stromal', 'Inflammatory/secretory', 'Interferon', 'Myofibroblast', 'Oxidative-stress', 'Proliferating', 'Senescent/stressed', 'Tendon'])


,source,target,coef_mean,coef_abs,p,-logp
46037,Klf2,Tagln,1.466639,1.466639,7.947532e-18,17.099768
46036,Junb,Tagln,-1.357715,1.357715,2.052934e-12,11.687625
1654,Klf2,Acta2,1.353102,1.353102,8.213034e-23,22.085496
7467,Mafk,Ccl7,1.042792,1.042792,1.349344e-05,4.869877
12096,Ebf1,Cxcl12,0.987530,0.987530,1.935671e-16,15.713168
38847,Ets1,Ptx3,0.982877,0.982877,2.203670e-18,17.656853
31421,Junb,Mt1,0.958568,0.958568,6.012557e-17,16.220941
29787,Ebf1,Malat1,0.949561,0.949561,2.477575e-17,16.605973
42215,Ebf1,Sfrp2,0.932576,0.932576,1.169062e-20,19.932162
13185,Myc,Dio3,0.927869,0.927869,2.344049e-18,17.630033


In [ ]:
import anndata as ad

# transfer the aging score computed in the ageing-axis notebook
adata_scored = ad.read_h5ad("/content/drive/MyDrive/roux_project/msc_aging_scored.h5ad")
print("order matches?", (oracle.adata.obs_names == adata_scored.obs_names).all())
oracle.adata.obs['aging_score'] = adata_scored.obs['aging_score_datadriven'].values
print("aging score attached, mean:", oracle.adata.obs['aging_score'].mean().round(3))

oracle.to_hdf5("/content/drive/MyDrive/roux_project/oracle_roux_v2.celloracle.oracle")
links.to_hdf5("/content/drive/MyDrive/roux_project/links_roux_v2.celloracle.links")
print("saved oracle and links")

order matches? True
aging score attached, mean: 0.758
saved oracle and links


In [ ]:
oracle.fit_GRN_for_simulation(alpha=10, use_cluster_specific_TFdict=True)

import numpy as np
for f in ['Klf4', 'Myc', 'Sox2', 'Pou5f1']:
    if f in oracle.adata.var_names:
        vals = np.asarray(oracle.adata[:, f].layers['imputed_count']).flatten()
        print(f"{f}: in network ✓ | max={vals.max():.2f} | mean={vals.mean():.3f} | %expressing={np.mean(vals>0)*100:.0f}%")
    else:
        print(f"{f}: not in network ✗")

  0%|          | 0/9 [00:00<?, ?it/s]

Klf4: in network ✓ | max=3.11 | mean=1.040 | %expressing=100%
Myc: in network ✓ | max=1.83 | mean=0.447 | %expressing=100%
Sox2: in network ✓ | max=0.67 | mean=0.021 | %expressing=38%
Pou5f1: in network ✓ | max=0.10 | mean=0.013 | %expressing=62%
